In [ ]:
!pip install transformers accelerate bitsandbytes

In [123]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch
import random

### Load a pretrained model

In [124]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")

Using device: mps


In [125]:
model_name = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_name) # just dictionary lookups, CPU is fine
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to(device) # default is float32; float16 halves the memory

print(sum(p.numel() for p in model.parameters())/1e6, "M parameters") # p.numel() gives the number of elements in a tensor

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 11280.24it/s]


125.239296 M parameters


### Tokenize data with proper loss masking


In [126]:
ds = load_dataset("tatsu-lab/alpaca")

In [127]:
def format_sample(row):
    if row["input"]:
        prompt = f"Instruction: {row["instruction"]}\nInput: {row["input"]}\nResponse:\n"
    else:
        prompt = f"Instruction: {row["instruction"]}\nResponse:\n"
    response = row["output"]
    return prompt, response

In [128]:
tokenizer("hello world")

{'input_ids': [2, 42891, 232], 'attention_mask': [1, 1, 1]}

In [129]:
def tokenize_sample(row, max_length=512):
    prompt, response = format_sample(row)
    full = prompt + response

    full_ids = tokenizer(full)["input_ids"]
    prompt_ids = tokenizer(prompt)["input_ids"]

    prompt_len = len(prompt_ids)

    # cap to max length
    full_ids = full_ids[:max_length]

    x = torch.tensor(full_ids[:-1])
    y = torch.tensor(full_ids[1:])
    y[:prompt_len-1] = -100 # mask prompt

    # skip if response is fully truncated
    if (y!=-100).sum() == 0:
        return None
    return x, y


In [130]:
tokenizer.eos_token, tokenizer.eos_token_id

('</s>', 2)

In [131]:
tokenizer.pad_token, tokenizer.pad_token_id

('<pad>', 1)

In [132]:
model.eval()

prompt = "Instruction: What is 2+2?.\nResponse:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device) # (B, T) => (1, T)

with torch.no_grad():
    out = model.generate(
        **inputs,
        temperature = 0.7,
        do_sample = True,
    )

In [133]:
response_idx = inputs["input_ids"][0].shape[0]
print(response_idx)

15


In [134]:
model.eval()

prompt = "Instruction: What is 2+2?.\nResponse:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device) # (B, T) -> (1, T)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=100, # default 35
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print("PRETRAINED (before SFT):\n")
response = tokenizer.decode(out[0][response_idx:], skip_special_tokens=False)
print(response)

PRETRAINED (before SFT):

To ask your student to choose the one that best suits you, you must be able to answer 2 questions.

To answer 2 questions, the student must be able to answer two questions correctly.

To answer 2 questions correctly, you must be able to answer 2 questions correctly.

To answer 2 questions correctly, you must be able to answer 2 questions correctly.

To answer 2 questions correctly, you must be able to answer 2 questions correctly.

To answer 2 questions


In [135]:
ds["train"], type(ds["train"]) # Apache Arrow format

(Dataset({
     features: ['instruction', 'input', 'output', 'text'],
     num_rows: 52002
 }),
 datasets.arrow_dataset.Dataset)

In [136]:
l = list(ds["train"])[:3]

In [ ]:
sft_list = list(ds["train"])
random.shuffle(sft_list)
tokenized = [tokenize_sample(row) for row in sft_list] # list
print(f"Total samples: {len(tokenized)}")
tokenized = [t for t in tokenized if t is not None] # filter out truncated samples
print(f"Valid samples: {len(tokenized)}")

# tokenized = [
#     (x0, y0),   # tuple from sample 0 — x0 shape: (T0,), y0 shape: (T0,)
#     (x1, y1),   # tuple from sample 1 — x1 shape: (T1,), y1 shape: (T1,)
# ]

print(tokenized[0][0].shape, tokenized[0][1].shape) # (x, y)

Total samples: 52002
Valid samples: 51973
torch.Size([28]) torch.Size([28])


In [155]:
def get_batch(data, batch_size=8):
    samples = random.sample(data, batch_size) # [(xi, yi), (xj, yj), ...]
    xs, ys = zip(*samples) # [(xi, xj, ...), (yi, yj, ...)]

    max_len = max(x.shape[0] for x in xs)
    xs = torch.stack([torch.nn.functional.pad(x, (0, max_len-x.size(0)), value=tokenizer.pad_token_id) for x in xs])
    ys = torch.stack([torch.nn.functional.pad(y, (0, max_len-y.size(0)), value=-100) for y in ys])

    return xs.to(device), ys.to(device)

In [156]:
import math

def get_lr(step, max_iters=2000, min_lr=1e-6, max_lr=2e-5):
    ratio = step / max_iters
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * ratio))

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
max_iters = 2000


In [ ]:
model.train()
for step in range(max_iters):
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr
    
    x, y = get_batch(tokenized)
    outputs = model(input_ids=x, labels=y) # model(...) calls the model's forward method; x: (B, T), y: (B, T)
    loss = outputs.loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}, LR: {lr:.2e}")

Step 0, Loss: nan, LR: 2.00e-05


In [ ]:
torch.save(model.state_dict(), "./opt_sft.pt")
print("Model saved to opt_sft.pt")

: 

: 

In [ ]:
# model.eval()
# prompt = "Instruction: List 3 benefits of exercise.\nResponse:\n"
# inputs = tokenizer(prompt, return_tensors="pt").to(device)

# with torch.no_grad():
#     out = model.generate(
#         **inputs,
#         max_new_tokens  = 200,
#         temperature     = 0.7,
#         do_sample       = True,
#         pad_token_id    = tokenizer.eos_token_id,
#     )

# # only print response, not prompt
# response = out[0][inputs["input_ids"].shape[1]:]
# print(tokenizer.decode(response, skip_special_tokens=True))

: 

In [ ]:
model.eval()

prompt = "Instruction: List 3 benefits of exercise.\nResponse:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=200
        
    )